# Sample Analysis
Place your analysis notebooks here.

In [ ]:
"""Recursive Floyd-Warshall."""

import sys
from typing import List

def floyd_warshall(g: List[List[float]], k: int = 0, i: int = 0, j: int = 0) -> List[List[float]]:
    """Recursive Floyd-Warshall algorithm."""
    n = len(g)
    if k >= n: return g
    if i >= n: return floyd_warshall(g, k + 1, 0, 0)
    if j >= n: return floyd_warshall(g, k, i + 1, 0)
    if g[i][k] + g[k][j] < g[i][j]: g[i][j] = g[i][k] + g[k][j]
    return floyd_warshall(g, k, i, j + 1)

def recursive_floyd(graph: List[List[float]]) -> List[List[float]]:
    """Wrapper for recursive Floyd-Warshall."""
    if not graph: return []
    if any(len(row) != len(graph) for row in graph): 
        raise ValueError("Matrix must be square")
    dist = [row[:] for row in graph]
    return floyd_warshall(dist, 0, 0, 0)

def print_matrix(m: List[List[float]]) -> None:
    """Print distance matrix."""
    for row in m:
        print("\t".join("INF" if x == sys.float_info.max else f"{x:.1f}" for x in row))

In [ ]:
"""Iterative Floyd-Warshall implementation."""

import sys
from typing import List

INF = sys.float_info.max

def floyd_warshall(graph: List[List[float]]) -> List[List[float]]:
    """Iterative Floyd-Warshall algorithm with three nested loops."""
    n = len(graph)
    dist = [row[:] for row in graph]  # Create working copy
    
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    
    return dist

def print_matrix(m: List[List[float]]) -> None:
    """Print distance matrix with INF handling."""
    for row in m:
        print("\t".join("INF" if x == INF else f"{x:.1f}" for x in row))

In [ ]:
import sys, os, unittest
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
from recursion.recursive_floyd import recursive_floyd
from recursion.iterative_floyd import floyd_warshall as iterative_floyd

INF = sys.float_info.max

class TestRecursiveFloyd(unittest.TestCase):
    def setUp(self):
        self.g1=[[0,5,INF],[INF,0,2],[3,INF,0]]
        self.e1=[[0,5,7],[5,0,2],[3,8,0]]
        self.g2=[[0,3,INF,7],[8,0,2,INF],[5,INF,0,1],[2,INF,INF,0]]
        self.e2=[[0,3,5,6],[5,0,2,3],[3,6,0,1],[2,5,7,0]]
    
    def test_simple(self): self.assertEqual(recursive_floyd(self.g1), self.e1)
    def test_four(self): self.assertEqual(recursive_floyd(self.g2), self.e2)
    def test_empty(self): self.assertEqual(recursive_floyd([]), [])
    def test_single(self): self.assertEqual(recursive_floyd([[0]]), [[0]])
    
    def test_compare(self):
        r=recursive_floyd(self.g2); i=iterative_floyd(self.g2)
        for x in range(4): 
            for y in range(4): self.assertAlmostEqual(r[x][y], i[x][y])
    
    def test_invalid(self):
        with self.assertRaises(ValueError): recursive_floyd([[0,1],[0]])

if __name__ == "__main__": unittest.main()

In [ ]:
import time, sys, os, random, tracemalloc
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
from recursion.recursive_floyd import recursive_floyd
from recursion.iterative_floyd import floyd_warshall as iterative_floyd

INF = sys.float_info.max
def gen_graph(n, d=0.3):
    g = [[INF]*n for _ in range(n)]
    for i in range(n):
        g[i][i] = 0
        for j in range(n):
            if i!=j and random.random()<d: g[i][j]=random.uniform(0.1,10)
    return g
def time_func(f, g):
    start=time.perf_counter(); r=f([x[:] for x in g]); return time.perf_counter()-start, r
def memory_func(f, g):
    tracemalloc.start(); r=f([x[:] for x in g]); _,peak=tracemalloc.get_traced_memory(); tracemalloc.stop()
    return peak, r

def equal(m1,m2,t=1e-10):
    for i in range(len(m1)):
        for j in range(len(m1)):
            if abs(m1[i][j]-m2[i][j])>t and (m1[i][j]==INF and m2[i][j]==INF): continue
            else: return False
    return True

def run_tests():
    print("Size\tIter(s)\tRec(s)\tRatio\tMatch")
    for n in range(2,9):
        g=gen_graph(n)
        it,t1=time_func(iterative_floyd,g); rt,t2=time_func(recursive_floyd,g)
        print(f"{n}\t{t1:.6f}\t{t2:.6f}\t{t2/t1:.2f}x\t{equal(it,rt)}")
if __name__=="__main__": run_tests()